# Kubernetes Deployment and Autoscaling
Companion notebook for the **Kubernetes Deployment and Autoscaling** section of Chapter 12.

**Pod** (one instance), **Deployment** (replica count + rollouts), **Service** (internal networking), **Ingress** (external routes). This notebook loads and validates the real `k8s-deployment.yaml` shipped alongside it -- the book's manifest, unchanged.

In [1]:
print(open("k8s-deployment.yaml").read())

apiVersion: apps/v1
kind: Deployment
metadata:
  name: agent-api
spec:
  replicas: 2
  selector:
    matchLabels:
      app: agent-api
  template:
    metadata:
      labels:
        app: agent-api
    spec:
      containers:
        - name: agent-api
          image: myrepo/agent-api:1.0.0
          ports:
            - containerPort: 8080
          resources:
            requests:
              cpu: "500m"
              memory: "1Gi"
            limits:
              cpu: "1"
              memory: "2Gi"



In [2]:
import yaml

manifest = yaml.safe_load(open("k8s-deployment.yaml"))
container = manifest["spec"]["template"]["spec"]["containers"][0]
requests = container["resources"]["requests"]
limits = container["resources"]["limits"]

print("replicas:", manifest["spec"]["replicas"])
print("image:   ", container["image"])
print("requests:", requests)
print("limits:  ", limits)

replicas: 2
image:    myrepo/agent-api:1.0.0
requests: {'cpu': '500m', 'memory': '1Gi'}
limits:   {'cpu': '1', 'memory': '2Gi'}


**Requests too low** risks the scheduler overpacking a node with agent pods. **Limits too tight** risks throttling or restarts on a long prompt or a bursty generation. A small sanity check -- limits should never sit *below* requests, and shouldn't be so far above them that a burst can starve its node-mates:

In [3]:
def parse_cpu(value: str) -> float:
    return float(value[:-1]) / 1000 if value.endswith("m") else float(value)


def check_resource_sanity(requests: dict, limits: dict, max_burst_ratio: float = 4.0) -> list[str]:
    problems = []
    req_cpu, lim_cpu = parse_cpu(requests["cpu"]), parse_cpu(limits["cpu"])
    if lim_cpu < req_cpu:
        problems.append("cpu limit is below cpu request")
    if lim_cpu > req_cpu * max_burst_ratio:
        problems.append(f"cpu limit is more than {max_burst_ratio}x the request -- risk of starving other pods")
    return problems


print("resource sanity issues:", check_resource_sanity(requests, limits))

resource sanity issues: []


**Autoscaling on the right signal**: CPU/memory are a starting point, but agent pods can be busy waiting on a model API with low CPU usage the whole time. Custom metrics -- queue depth, p95 latency, in-flight sessions, tokens/sec -- are usually better. A small decision function using queue depth as the clearest signal for a worker-style agent, the book's own recommendation:

In [4]:
def recommend_replica_count(current_replicas: int, queue_depth: int, target_per_replica: int = 20,
                             min_replicas: int = 2, max_replicas: int = 20) -> int:
    import math
    needed = math.ceil(queue_depth / target_per_replica)
    return max(min_replicas, min(max_replicas, max(current_replicas, needed)))


print("light load  (queue_depth=15):", recommend_replica_count(2, queue_depth=15))
print("heavy load  (queue_depth=180):", recommend_replica_count(2, queue_depth=180))
print("already scaled, load drops -- HPA would scale back down separately, not shown here")

light load  (queue_depth=15): 2
heavy load  (queue_depth=180): 9
already scaled, load drops -- HPA would scale back down separately, not shown here


For rolling updates: readiness probes so new pods only get traffic after warm-up, enough termination time for in-flight requests to finish, and idempotent workers for long-running jobs so a pod restart can't duplicate work.